
# 練習問題: reduction で総和の競合を解消する

## 目標

`reduction(+:s)` 句を1つ追加するだけで, 複数のスレッドが同じ変数 `s` に同時に加算しようとして起こる競合 (data race) を解消し, 正しい総和が得られることを体験する.

## 課題

`reduction_sum.cpp` (または `reduction_sum.f90`) は, 区間 [a, b] で `1/(1+x*x)` を数値積分するコードである.
このループを単純に並列化すると, 全スレッドが共有変数 `s` に同時に `s += ...` を行うため競合 (data race) が起き, スレッド数が2以上だと結果が間違ったり実行ごとに変わったりする.

コメント `TODO` の箇所で **`reduction` を用いて並列化せよ**. 各スレッドが部分和を持ち寄って正しい総和を得るようになる.

- C++: `#pragma omp parallel for reduction(+:s)`
- Fortran: `!$omp parallel do private(x) reduction(+:s)` … `!$omp end parallel do`

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore reduction_sum.cpp -o reduction_sum.exe

# Fortran
nvfortran -fast -mp=multicore reduction_sum.f90 -o reduction_sum.exe
```

```
OMP_NUM_THREADS=4 ./reduction_sum.exe
```

## 期待される結果

`[a, b] = [0, 1]` での `1/(1+x*x)` の積分は `π/4 ≒ 0.785398` である.

- `reduction(+:s)` を入れる前 (`OMP_NUM_THREADS=4`) は, 結果が 0.785398 から大きくずれ, 実行のたびに値が変わることがある (競合).
- `reduction(+:s)` を追加すると, スレッド数によらず常に `s = 0.785398` 付近になる.

```
s = 0.785398
```



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:reduction_sum.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:reduction_sum.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ reduction_sum.cpp
#include <cstdio>
#include <cstdlib>

double int_inv_1_x2(double a, double b, long n) {
  double s = 0.0;
  double dx = (b - a) / (double)n;
  // TODO: 下のループを reduction(+:s) を用いて並列化し, 総和の競合を解消せよ.
  for (long i = 0; i < n; i++) {
    double x = a + i * dx;
    s += 1 / (1 + x * x);
  }
  return s * dx;
}

int main(int argc, char ** argv) {
  double a = (argc > 1 ? atof(argv[1]) : 0.0);
  double b = (argc > 2 ? atof(argv[2]) : 1.0);
  long n   = (argc > 3 ? atol(argv[3]) : 1000L * 1000L * 1000L);
  double s = int_inv_1_x2(a, b, n);
  printf("s = %f\n", s);
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore reduction_sum.cpp -o reduction_sum_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./reduction_sum_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./reduction_sum_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./reduction_sum_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:reduction_sum.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ reduction_sum.f90
module integral_mod
contains
  function int_inv_1_x2(a, b, n) result(s)
    real(8), intent(in) :: a, b
    integer(8), intent(in) :: n
    real(8) :: s, dx, x
    integer(8) :: i
    s = 0.0d0
    dx = (b - a) / real(n, 8)
    ! TODO: 下のループを reduction(+:s) を用いて並列化し, 総和の競合を解消せよ.
    do i = 0, n - 1
       x = a + i * dx
       s = s + 1 / (1 + x * x)
    end do
    ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
    s = s * dx
  end function int_inv_1_x2
end module integral_mod

program reduction_sum
  use integral_mod
  character(len=64) :: arg
  real(8) :: a, b, s
  integer(8) :: n
  a = 0.0d0; b = 1.0d0; n = 1000_8 * 1000_8 * 1000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) a
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) b
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) n
  end if
  s = int_inv_1_x2(a, b, n)
  print "(a,f0.6)", "s = ", s
end program reduction_sum

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore reduction_sum.f90 -o reduction_sum_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./reduction_sum_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./reduction_sum_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./reduction_sum_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:reduction_sum.f90}